In [106]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [107]:
pc_folder_path = 'L:/2026 Pilar Plant Files/Position Control'

# --- adjust date ---    
file_date = '2026-08-29'  
# -------------------

# get all matching files    
pc_files = glob.glob(os.path.join(pc_folder_path, f'*{file_date}*.xlsx'))

print(f"Found {len(pc_files)} files matching date {file_date}")

site_pc_list = []

for file in pc_files:    
    filename = os.path.basename(file)    
      
    try:  
        # extract site (first 5 characters of filename)    
        site = filename[:5]    
          
        # read second tab, skip first 12 rows    
        file_data = pd.read_excel(file, sheet_name='PositionDetail', skiprows=12)    
          
        # add site and date columns    
        file_data['BU'] = site    
        file_data['Date'] = file_date  

        f_file_data = file_data[['Rpt Dept', 'Rpt Dept Desc', 'Job Code', 'Jobcode Title', 'Job Function / Family', 'Posn Number',   
                                'Posn Status', 'Filled/ Open', 'Incumbent Name', 'Emplid', 'Incumbent Status',  
                                'Reg/ Temp', 'Posn Type', 'Filled Hrs', 'Filled FTE', 'Open Hrs', 'Open FTE', 'BU', 'Date']]

        fc_file_data = f_file_data[f_file_data['Rpt Dept'].notna()]  
          
        site_pc_list.append(fc_file_data)  
        print(f"Processed: {filename}")  
      
    except PermissionError:  
        print(f"SKIPPED (permission denied): {filename}")  
        continue

# combine all into one dataframe    
if site_pc_list:    
    position_control = pd.concat(site_pc_list, ignore_index=True)    
    print(f"\nTotal rows: {len(position_control)}")    
else:    
    print(f"No files found for date: {file_date}")  

Found 4 files matching date 2026-08-29
Processed: GVCCC POSNEXEC GVCCC_EXEC 2026-08-29.xlsx
Processed: LENOX POSNEXEC LENOX_EXEC 2026-08-29.xlsx
Processed: MEETH POSNEXEC MEETH_EXEC 2026-08-29.xlsx
SKIPPED (permission denied): ~$LENOX POSNEXEC LENOX_EXEC 2026-08-29.xlsx

Total rows: 6272


In [108]:
position_control['filled_active'] = np.where(position_control['Incumbent Status'] == 'A', position_control['Filled FTE'], 0) 
position_control['loa'] = np.where(position_control['Incumbent Status'].isin(['L', 'P']), position_control['Filled FTE'], 0)
position_control['open'] = np.where(position_control['Filled/ Open'] == 'Open', position_control['Open FTE'], 0)
position_control['total_ftes'] = position_control['filled_active'] + position_control['loa'] + position_control['open']

pc_values = position_control[position_control['total_ftes'] != 0]
pc_full = pc_values.groupby(['BU', 'Date', 'Rpt Dept', 'Job Code'])[['filled_active', 'loa', 'open', 'total_ftes']].sum().reset_index()

display(pc_full[['filled_active', 'loa', 'open', 'total_ftes']].sum())
display(pc_full[['Rpt Dept', 'Job Code']].drop_duplicates().count())

filled_active    3928.74
loa               216.22
open              520.10
total_ftes       4665.06
dtype: float64

Rpt Dept    1018
Job Code    1018
dtype: int64

In [109]:
req_path_name = 'L:/2026 Pilar Plant Files/Requisition Reports/'
req_file_name = 'NYC Req Reports 9.3.2026.xlsx'
pending_req_tab = 'Pending'
approved_req_tab = 'Open'

file = req_path_name + req_file_name

# read two tabs into separate dataframes  
pending_reqs = pd.read_excel(file, sheet_name=pending_req_tab) 
approved_reqs = pd.read_excel(file, sheet_name=approved_req_tab)   

approved_reqs_site_spec = approved_reqs[approved_reqs['BUSINESS UNIT'].isin(['LENOX', 'MEETH', 'GVCCC'])]
approved_reqs_full = approved_reqs_site_spec.groupby(['BUSINESS UNIT', 'DEPARTMENT NUMBER', 'JOB CODE'])['FTE'].sum().reset_index()

pending_reqs_site_spec = pending_reqs[pending_reqs['BUSINESS UNIT'].isin(['LENOX', 'MEETH', 'GVCCC'])]
pending_reqs_full = pending_reqs_site_spec.groupby(['BUSINESS UNIT', 'DEPARTMENT NUMBER', 'JOB CODE', 'NEW REPLACE'])['FTE'].sum().reset_index()

pending_reqs_piv = pending_reqs_full.pivot_table(  
    index=['BUSINESS UNIT', 'DEPARTMENT NUMBER', 'JOB CODE'],       # your row identifier  
    columns='NEW REPLACE',                                                                          # whatever labels exist, they become columns  
    values='FTE',                                                                                   # the values to fill in  
    aggfunc='sum'  
).reset_index()

pending_reqs_piv.columns.name = None 

In [110]:
display(approved_reqs_full[['FTE']].sum())

FTE    129.47619
dtype: float64

In [111]:

display(pc_full[['Rpt Dept', 'Job Code']].drop_duplicates().count())
display(approved_reqs_full[['DEPARTMENT NUMBER', 'JOB CODE']].drop_duplicates().count())
display(pending_reqs_piv[['DEPARTMENT NUMBER', 'JOB CODE']].drop_duplicates().count())

Rpt Dept    1018
Job Code    1018
dtype: int64

DEPARTMENT NUMBER    118
JOB CODE             118
dtype: int64

DEPARTMENT NUMBER    35
JOB CODE             35
dtype: int64

In [112]:
pc_app_reqs = pd.merge(pc_full, approved_reqs_full, how='outer', left_on=['Rpt Dept', 'Job Code'], right_on=['DEPARTMENT NUMBER', 'JOB CODE'])

pc_app_reqs['dept_id'] = np.where(pc_app_reqs['Rpt Dept'].notna(), pc_app_reqs['Rpt Dept'], pc_app_reqs['DEPARTMENT NUMBER'])
pc_app_reqs['bu'] = np.where(pc_app_reqs['BU'].notna(), pc_app_reqs['BU'], pc_app_reqs['BUSINESS UNIT'])  
pc_app_reqs['job_code'] = np.where(pc_app_reqs['Job Code'].notna(), pc_app_reqs['Job Code'], pc_app_reqs['JOB CODE'])  
pc_app_reqs['approved_ftes'] = pc_app_reqs['FTE']

pc_app_reqs_merged = pc_app_reqs[['bu', 'dept_id', 'job_code', 'filled_active', 'loa', 'open', 'total_ftes','approved_ftes']]

display(pc_app_reqs_merged.head())

,bu,dept_id,job_code,filled_active,loa,open,total_ftes,approved_ftes
0,LENOX,15600000.0,103442.0,0.5,0.0,0.0,0.5,NaN
1,LENOX,15600000.0,107015.0,1.0,0.0,0.0,1.0,NaN
2,LENOX,15600000.0,200118.0,0.5,0.0,0.0,0.5,NaN
3,LENOX,15600000.0,200222.0,0.9,0.0,0.0,0.9,NaN
4,LENOX,15600000.0,201068.0,1.0,0.0,1.0,2.0,NaN


In [113]:
pc_app_pend_reqs = pd.merge(pc_app_reqs_merged, pending_reqs_piv, how='outer', left_on=['dept_id', 'job_code'], right_on=['DEPARTMENT NUMBER', 'JOB CODE'])

pc_app_pend_reqs['dept_id'] = np.where(pc_app_pend_reqs['dept_id'].notna(), pc_app_pend_reqs['dept_id'], pc_app_pend_reqs['DEPARTMENT NUMBER'])
pc_app_pend_reqs['bu'] = np.where(pc_app_pend_reqs['bu'].notna(), pc_app_pend_reqs['bu'], pc_app_pend_reqs['BUSINESS UNIT'])  
pc_app_pend_reqs['job_code'] = np.where(pc_app_pend_reqs['job_code'].notna(), pc_app_pend_reqs['job_code'], pc_app_pend_reqs['JOB CODE'])  
pc_app_pend_reqs['new_ftes'] = pc_app_pend_reqs['NEW']
pc_app_pend_reqs['replacement_ftes'] = pc_app_pend_reqs['REPLACE']
# might need position change column tbd

pc_reqs_full = pc_app_pend_reqs.fillna(0)

pc_reqs_full['total_req_ftes'] = pc_reqs_full['approved_ftes'] + pc_reqs_full['new_ftes'] + pc_reqs_full['replacement_ftes']

pc_reqs_merged = pc_reqs_full[['bu', 'dept_id', 'job_code', 'filled_active', 'loa', 'open', 'total_ftes', 'approved_ftes', 'new_ftes', 'replacement_ftes', 'total_req_ftes']]

display(pc_reqs_merged)

pc_reqs_merged[['dept_id', 'job_code']].drop_duplicates().count()

,bu,dept_id,job_code,filled_active,loa,open,total_ftes,approved_ftes,new_ftes,replacement_ftes,total_req_ftes
0,LENOX,15600000.0,103442.0,0.5,0.0,0.0,0.5,0.0,0.0,0.0,0.0
1,LENOX,15600000.0,107015.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,LENOX,15600000.0,200118.0,0.5,0.0,0.0,0.5,0.0,0.0,0.0,0.0
3,LENOX,15600000.0,200222.0,0.9,0.0,0.0,0.9,0.0,0.0,0.0,0.0
4,LENOX,15600000.0,201068.0,1.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1034,GVCCC,74002562.0,904195.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1035,GVCCC,74002564.0,108102.0,7.0,1.0,0.0,8.0,0.0,0.0,0.0,0.0
1036,GVCCC,74002564.0,115242.0,2.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0
1037,GVCCC,74002598.0,204038.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


dept_id     1039
job_code    1039
dtype: int64

In [105]:
# pull xwalk reference file
xwalk = pd.read_excel("C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/dept_jc_lookup_table.xlsx")

# check for duplicates in crosswalk  
dupes = xwalk[xwalk.duplicated(subset=['dept', 'jc'], keep=False)]
# print dupes if found
if not dupes.empty:  
    print(f"WARNING: {len(dupes)} duplicate dept/jc rows found in crosswalk:")  
    display(dupes.sort_values(['dept', 'jc']))

dept_lookup = xwalk[['vp', 'director', 'dept', 'dept_desc']].drop_duplicates()

pc_reqs_dept = pd.merge(dept_lookup, pc_reqs_merged, how='right', left_on='dept', right_on='dept_id')

# remove corporate retained and employee health services
pc_depts_f = pc_reqs_dept[~pc_reqs_dept['dept_desc'].str.contains('Corporate Retained|Corp Retained|Employee Health Svcs', case=False, na=False)]  
display(pc_depts_f)
display(pc_depts_f[['filled_active', 'loa', 'open', 'total_ftes', 'approved_ftes', 'new_ftes', 'replacement_ftes', 'total_req_ftes']].sum())


,vp,director,dept,dept_desc,bu,dept_id,job_code,filled_active,loa,open,total_ftes,approved_ftes,new_ftes,replacement_ftes,total_req_ftes
0,"Jurik, Christopher","Debacker, Johanna M",15600000.0,Hospital Administration,LENOX,15600000.0,103442.0,0.5,0.0,0.0,0.5,0.0,0.0,0.0,0.0
1,"Jurik, Christopher","Debacker, Johanna M",15600000.0,Hospital Administration,LENOX,15600000.0,107015.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,"Jurik, Christopher","Debacker, Johanna M",15600000.0,Hospital Administration,LENOX,15600000.0,200118.0,0.5,0.0,0.0,0.5,0.0,0.0,0.0,0.0
3,"Jurik, Christopher","Debacker, Johanna M",15600000.0,Hospital Administration,LENOX,15600000.0,200222.0,0.9,0.0,0.0,0.9,0.0,0.0,0.0,0.0
4,"Jurik, Christopher","Debacker, Johanna M",15600000.0,Hospital Administration,LENOX,15600000.0,201068.0,1.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1034,"Cohen, Jill","Yoon, Shin Hae",74002562.0,Pos-Central Services,GVCCC,74002562.0,904195.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1035,"Cohen, Jill","Yoon, Shin Hae",74002564.0,POS - Perianesthesia,GVCCC,74002564.0,108102.0,7.0,1.0,0.0,8.0,0.0,0.0,0.0,0.0
1036,"Cohen, Jill","Yoon, Shin Hae",74002564.0,POS - Perianesthesia,GVCCC,74002564.0,115242.0,2.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0
1037,"Baker, Daniel","Dempsey, Allison",74002598.0,EXE Home,GVCCC,74002598.0,204038.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


filled_active       3928.740000
loa                  216.220000
open                 520.100000
total_ftes          4665.060000
approved_ftes        129.476190
new_ftes               5.526667
replacement_ftes      30.300000
total_req_ftes       165.302857
dtype: float64